# Hot swap a model in the live Serve app

This notebook changes the checkpoint of one model deployment in a running Anyscale Service, with no
restart and no new cluster. It sends the same SMILES before and after the swap, and shows the model
version that answered each request.

**How it works**

- `Boltz2Live` and `FineTunedLive` in `serve_viewer.py` take `{"version", "checkpoint"}` as their
  Ray Serve `user_config`.
- `anyscale.service.deploy(..., in_place=True)` with a config that changes only `user_config` calls
  `reconfigure()` on the running replicas. Ray Serve does not restart them.
- Each prediction runs the model as a new subprocess, so the next request already uses the new
  checkpoint. Each version has its own cache folder, so the old model's cached poses never come back.

**Before you start**

- Run this notebook in a Workspace, from the repository folder. It deploys `service.yaml` from here.
- The data must be in `/mnt/shared_storage/aiddd` (README Step 2).
- Do not change files in this folder between the first deploy and the swap. A code change makes the
  update restart every replica. Notebooks are excluded from the upload, so editing this one is safe.

In [ ]:
import json, os, shutil, time
from pathlib import Path

import anyscale
import requests
from anyscale.service.models import ServiceConfig

SHARED = Path(os.environ.get("AIDDD_SHARED", "/mnt/shared_storage/aiddd"))

# Your own Service. "aiddd-viewer" is the shared workshop Service: never use it for practice.
SERVICE_NAME = "aiddd-viewer-geoff"
assert SERVICE_NAME != "aiddd-viewer", "pick your own Service name for practice runs"

# A molecule to send before and after the swap. Pick one where the old model is far from the crystal.
SMILES = "CNC(=O)c1nn(C)c2ccccc12"

# The deployment to swap and its new user_config.
DEPLOYMENT = "Boltz2Live"   # or "FineTunedLive" for the OpenFold3 pose model
BASE = {"version": "base", "checkpoint": None}
NEW = {"version": "ft-v1", "checkpoint": str(SHARED / "checkpoints" / "boltz2_ft_v1.ckpt")}

for p in (SHARED / "boltz_cache", SHARED / "results"):
    assert p.exists(), f"missing {p}: copy the data into /mnt/shared_storage/aiddd first"

## Practice checkpoint

There is no fine-tuned Boltz-2 checkpoint yet. For practice, this cell copies the released Boltz-2
weights under a new name (2.3 GB, about a minute). The predictions do not change, but the version and
checkpoint in the response do, which proves the swap. Skip this cell when a real checkpoint exists.

In [ ]:
ckpt = Path(NEW["checkpoint"])
if DEPLOYMENT == "Boltz2Live" and not ckpt.exists():
    ckpt.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy(SHARED / "boltz_cache" / "boltz2_conf.ckpt", ckpt)
print(ckpt, f"{ckpt.stat().st_size / 1e9:.1f} GB")

## Helpers

`service_config()` is `service.yaml` with your Service name and, optionally, a `user_config` for each
deployment. `predict()` calls the Viewer's `/predict` endpoint, the same one the web page uses.

In [ ]:
def service_config(user_configs: dict) -> ServiceConfig:
    """service.yaml, renamed, with {deployment name: user_config} overrides."""
    cfg = ServiceConfig.from_yaml("service.yaml", name=SERVICE_NAME)
    deployments = [{"name": d, "user_config": uc} for d, uc in user_configs.items()]
    return cfg.options(applications=[{**app, "deployments": deployments} for app in cfg.applications])


def predict(smiles: str) -> dict:
    s = anyscale.service.status(name=SERVICE_NAME)
    headers = {"Authorization": f"Bearer {s.query_auth_token}"} if s.query_auth_token else {}
    r = requests.post(s.query_url.rstrip("/") + "/predict", json={"smiles": smiles},
                      headers=headers, timeout=600)
    return r.json()


def show(label: str, out: dict):
    """The model metadata and the main numbers of one /predict response."""
    m = out.get("metrics", {})
    print(f"--- {label}: status {out.get('status')}, {out.get('seconds', 0):.0f} s")
    print(json.dumps(out.get("models"), indent=2))
    print(f"pose ipTM {out.get('pose_iptm')}, affinity {m.get('affinity_pred_value')}, "
          f"P(binder) {m.get('affinity_probability_binary')}")

## 1. Start the Service with the base models

The first deploy starts a new cluster with two L4 nodes, which takes about 5 to 10 minutes. If the
Service is already running, this cell skips the deploy.

In [ ]:
try:
    state = anyscale.service.status(name=SERVICE_NAME).state
except Exception:   # no Service with this name yet
    state = None
if str(state) != "RUNNING":
    anyscale.service.deploy(service_config({DEPLOYMENT: BASE}), name=SERVICE_NAME)
    anyscale.service.wait(name=SERVICE_NAME, state="RUNNING", timeout_s=1800)
print(anyscale.service.status(name=SERVICE_NAME).query_url)

## 2. Before the swap

A new molecule takes one to three minutes. Repeat requests return from the cache.

In [ ]:
before = predict(SMILES)
show("before", before)

## 3. Hot swap

`in_place=True` reuses the running cluster. Only `user_config` changes, so Ray Serve calls
`reconfigure()` on the live replicas. The loop then sends the same SMILES until a response comes from
the new version, which is usually the first try.

In [ ]:
t0 = time.time()
anyscale.service.deploy(service_config({DEPLOYMENT: NEW}), in_place=True, name=SERVICE_NAME)
anyscale.service.wait(name=SERVICE_NAME, state="RUNNING", timeout_s=600)

key = "affinity" if DEPLOYMENT == "Boltz2Live" else "pose"
for _ in range(30):
    after = predict(SMILES)
    if (after.get("models") or {}).get(key, {}).get("version") == NEW["version"]:
        break
    time.sleep(5)
print(f"swap took {time.time() - t0:.0f} s, including the first prediction on the new version")
show("after", after)

## 4. Before and after

In [ ]:
import pandas as pd

def row(out):
    m, models = out.get("metrics", {}), out.get("models") or {}
    return {"pose model": f"{models.get('pose', {}).get('name')} {models.get('pose', {}).get('version')}",
            "affinity model": f"{models.get('affinity', {}).get('name')} {models.get('affinity', {}).get('version')}",
            "pose ipTM": out.get("pose_iptm"), "affinity": m.get("affinity_pred_value"),
            "P(binder)": m.get("affinity_probability_binary"), "seconds": out.get("seconds")}

pd.DataFrame({"before": row(before), "after": row(after)})

## 5. Swap back

The same in-place update, with the base `user_config`.

In [ ]:
anyscale.service.deploy(service_config({DEPLOYMENT: BASE}), in_place=True, name=SERVICE_NAME)

## 6. Stop the Service

The Service keeps two L4 nodes running until you stop it.

In [ ]:
# anyscale.service.terminate(name=SERVICE_NAME)